# Tutorial: xarray and Plotting

This notebook introduces [xarray](https://docs.xarray.dev/), a Python library for working with
labelled multi-dimensional arrays. xarray is particularly popular in the geosciences but is useful
any time your data has named dimensions (e.g. time, latitude, longitude).

**Topics covered**
1. Creating `DataArray` and `Dataset` objects
2. Inspecting and selecting data
3. Arithmetic and aggregation
4. Plotting with xarray's built-in `.plot()` interface
5. Customising plots with Matplotlib

**Prerequisites:** basic Python, NumPy and Matplotlib familiarity.

---

## 0. Installation

If xarray is not yet installed, run the following cell once:

In [ ]:
# Uncomment and run if needed
# !pip install xarray matplotlib numpy scipy

## 1. Imports

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Display plots inline in the notebook
%matplotlib inline

print(f"xarray version : {xr.__version__}")
print(f"numpy  version : {np.__version__}")

---
## 2. The `DataArray` — a labelled N-dimensional array

An `xr.DataArray` wraps a NumPy array and adds:
* **dimensions** – named axes (e.g. `'time'`, `'lat'`, `'lon'`)
* **coordinates** – tick values along each dimension
* **attributes** – metadata such as units and long names

In [ ]:
# --- 2.1  A simple 1-D DataArray ---
months = np.arange(1, 13)          # 1 … 12
temperature = np.array(
    [3.1, 4.5, 8.2, 13.6, 18.3, 22.1,
     24.8, 24.2, 19.7, 13.9, 8.0, 4.1]
)

da = xr.DataArray(
    data=temperature,
    dims=["month"],
    coords={"month": months},
    attrs={"units": "°C", "long_name": "Monthly mean surface temperature"},
)

print(da)

In [ ]:
# Key properties
print("Shape      :", da.shape)
print("Dimensions :", da.dims)
print("Coordinates:", list(da.coords))
print("Attributes :", da.attrs)

In [ ]:
# --- 2.2  A 2-D DataArray (latitude × longitude) ---
lats = np.linspace(-90, 90, 37)     # every 5 degrees
lons = np.linspace(-180, 180, 73)   # every 5 degrees

# Synthetic sea-surface temperature field
rng = np.random.default_rng(seed=42)
sst_data = 28 * np.cos(np.deg2rad(lats))[:, None] + rng.normal(0, 1, (37, 73))

sst = xr.DataArray(
    data=sst_data,
    dims=["lat", "lon"],
    coords={"lat": lats, "lon": lons},
    attrs={"units": "°C", "long_name": "Sea-surface temperature"},
)

print(sst)

---
## 3. The `Dataset` — a collection of `DataArray`s

A `Dataset` is a dictionary-like container that holds multiple `DataArray` variables sharing
the same dimensions/coordinates.

In [ ]:
# Build a small dataset with temperature and precipitation over months
precip = np.array([55, 42, 48, 45, 58, 62, 75, 70, 55, 60, 65, 60])

ds = xr.Dataset(
    {
        "temperature": xr.DataArray(
            temperature, dims=["month"], coords={"month": months},
            attrs={"units": "°C"}
        ),
        "precipitation": xr.DataArray(
            precip, dims=["month"], coords={"month": months},
            attrs={"units": "mm"}
        ),
    },
    attrs={"description": "Monthly climate normals"},
)

print(ds)

In [ ]:
# Access a variable
print(ds["temperature"])

# or using attribute access
print(ds.precipitation)

---
## 4. Indexing and Selection

xarray provides two main ways to index data:

| Method | Selector | What it does |
|--------|----------|--------------|
| `.isel()` | integer position | like NumPy `[i]` |
| `.sel()` | coordinate label | like a dictionary lookup |

In [ ]:
# Select by position (0-indexed)
print("First month (isel):", da.isel(month=0).values)

# Select by coordinate label
print("June (sel)      :", da.sel(month=6).values)

# Slice a range
print("Summer (Jun–Aug):", da.sel(month=slice(6, 8)).values)

In [ ]:
# Nearest-neighbour selection (useful for floating-point coordinates)
print("SST at lat≈30, lon≈45:")
print(sst.sel(lat=30, lon=45, method="nearest"))

In [ ]:
# Boolean (where) selection – mask values where temperature < 10 °C
cold_months = da.where(da < 10)
print(cold_months)

---
## 5. Arithmetic and Aggregation

Arithmetic on xarray objects propagates dimension labels automatically.

In [ ]:
# Unit conversion: °C → °F
temp_F = da * 9 / 5 + 32
temp_F.attrs["units"] = "°F"
print(temp_F)

In [ ]:
# Aggregation along a dimension
print("Annual mean temperature :", float(da.mean("month")), da.attrs.get("units", ""))
print("Hottest month index     :", int(da.argmax("month")))
print("Hottest month number    :", int(da["month"][da.argmax("month")]))

In [ ]:
# Aggregation on 2-D SST: zonal mean (average over all longitudes)
zonal_mean_sst = sst.mean(dim="lon")
print(zonal_mean_sst)

---
## 6. Plotting with xarray

xarray's `.plot()` method automatically labels axes using the array's dimension names and
attributes (units, long_name).

### 6.1 Line plot (1-D)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
da.plot(ax=ax, marker="o", color="tomato")
ax.set_title("Monthly Mean Surface Temperature")
ax.set_xticks(months)
ax.set_xticklabels(
    ["Jan","Feb","Mar","Apr","May","Jun",
     "Jul","Aug","Sep","Oct","Nov","Dec"]
)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 6.2 Bar chart using Matplotlib

In [ ]:
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].bar(months, ds["temperature"].values, color="salmon", label="Temperature (°C)")
axes[0].set_ylabel("Temperature (°C)")
axes[0].set_title("Monthly Climate Normals")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

axes[1].bar(months, ds["precipitation"].values, color="steelblue", label="Precipitation (mm)")
axes[1].set_ylabel("Precipitation (mm)")
axes[1].set_xlabel("Month")
axes[1].set_xticks(months)
axes[1].set_xticklabels(month_labels)
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### 6.3 Heatmap / `pcolormesh` (2-D)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sst.plot(ax=ax, cmap="RdBu_r", robust=True)
ax.set_title("Synthetic Sea-Surface Temperature")
plt.tight_layout()
plt.show()

### 6.4 Zonal mean profile

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
zonal_mean_sst.plot(ax=ax, y="lat", color="navy")
ax.set_title("Zonal Mean SST")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Latitude (°)")
ax.axvline(0, color="gray", linestyle="--")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### 6.5 Facet plots

When a dataset has multiple variables (or a third dimension), xarray can create small multiples
automatically.

In [ ]:
# Create a 3-D array: SST over 4 seasons
seasons = ["DJF", "MAM", "JJA", "SON"]
seasonal_sst = xr.DataArray(
    data=np.stack(
        [28 * np.cos(np.deg2rad(lats))[:, None] * factor + rng.normal(0, 0.5, (37, 73))
         for factor in [0.7, 0.9, 1.0, 0.8]],
        axis=0
    ),
    dims=["season", "lat", "lon"],
    coords={"season": seasons, "lat": lats, "lon": lons},
    attrs={"units": "°C", "long_name": "Sea-surface temperature"},
)

# xarray's facet plot – one panel per season
fg = seasonal_sst.plot(
    col="season",
    col_wrap=2,
    cmap="RdBu_r",
    robust=True,
    figsize=(12, 8),
)
fg.set_titles("{value}")
plt.suptitle("Seasonal SST", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Saving and Loading Data

xarray natively supports **NetCDF** (the standard format in climate science) and **Zarr**.

In [ ]:
# Save the dataset to a NetCDF file
ds.to_netcdf("/tmp/climate_normals.nc")

# Load it back
ds_loaded = xr.open_dataset("/tmp/climate_normals.nc")
print(ds_loaded)

---
## 8. Summary

| Concept | Key function / method |
|---------|----------------------|
| Create DataArray | `xr.DataArray(data, dims, coords, attrs)` |
| Create Dataset | `xr.Dataset({name: da, ...})` |
| Select by position | `.isel(dim=i)` |
| Select by label | `.sel(dim=value)` |
| Mask values | `.where(condition)` |
| Aggregation | `.mean()`, `.sum()`, `.max()`, … |
| Quick plot | `.plot()` |
| Save / load | `.to_netcdf()` / `xr.open_dataset()` |

**Next steps**
* Explore the [xarray documentation](https://docs.xarray.dev/) for advanced topics such as
  `groupby`, `resample`, and `apply_ufunc`.
* Try [cartopy](https://scitools.org.uk/cartopy/) for geographic map projections.
* Try [hvplot](https://hvplot.holoviz.org/) for interactive visualisations.